# Company Alias Dimension Loader

This notebook maintains the `warehouse.dim_company_alias` dimension table using standardized batch processing.

**Purpose**: Map company name variations to canonical company entities

**Key Features**:
* Links company aliases to canonical companies via company_sk
* Extracts original company names from semantic mapping layer
* Tracks match confidence and method for each alias
* Enables fuzzy company name lookup in fact table joins

**Architecture**:
- **Source**: Semantic layer company mappings (`workspace.semantic.sem_company_map`)
- **Reference**: Warehouse company dimension (`workspace.warehouse.dim_company`)
- **Target**: `workspace.warehouse.dim_company_alias`
- **Metadata**: `workspace.metadata.dim_company_alias_refresh_log`

**Processing Logic**:
1. Extract company name variations from `sem_company_map.explanation_json`
2. Link aliases to canonical companies via `dim_company`
3. Generate stable surrogate keys for new aliases
4. Merge into dimension with full audit trail

**Idempotency**: Safe to re-run; uses MERGE for upsert logic

In [0]:
dbutils.widgets.text("force_full_refresh", "false", "Force Full Refresh (true/false)")

force_full_refresh = dbutils.widgets.get("force_full_refresh").strip().lower() == "true"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DoubleType, DecimalType, TimestampType, BooleanType
from pyspark.sql.window import Window
from datetime import datetime
import json

CATALOG = "workspace"
SEMANTIC_SCHEMA = f"{CATALOG}.semantic"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Source and target tables
SOURCE_MAP_TABLE = f"{SEMANTIC_SCHEMA}.sem_company_map"
COMPANY_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_company"
TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_company_alias"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_company_alias_refresh_log"

# Generate run ID
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = F.current_timestamp()

print(f"Run ID: {run_id}")
print(f"Source table: {SOURCE_MAP_TABLE}")
print(f"Force full refresh: {force_full_refresh}")

In [0]:
# Define target table schema
target_schema = StructType([
    StructField("company_alias_sk", LongType(), False, metadata={"comment": "Surrogate key for company alias"}),
    StructField("alias_name", StringType(), False, metadata={"comment": "Company name variation"}),
    StructField("canonical_company_name", StringType(), False, metadata={"comment": "Canonical company name"}),
    StructField("company_sk", LongType(), False, metadata={"comment": "Foreign key to dim_company"}),
    StructField("company_match_method", StringType(), True, metadata={"comment": "Matching method used"}),
    StructField("match_confidence", DoubleType(), True, metadata={"comment": "Match confidence score"}),
    StructField("is_active", BooleanType(), False, metadata={"comment": "Is alias active"}),
    StructField("updated_at", TimestampType(), False, metadata={"comment": "Last update timestamp"})
])

# Check if table exists and has correct schema
table_exists = spark.catalog.tableExists(TARGET_TABLE)
if table_exists:
    # Check if schema matches
    existing_cols = set([field.name for field in spark.table(TARGET_TABLE).schema.fields])
    expected_cols = set([field.name for field in target_schema.fields])
    
    if existing_cols != expected_cols:
        print(f"Schema mismatch detected. Checking if table is empty...")
        row_count = spark.table(TARGET_TABLE).count()
        if row_count == 0:
            print(f"Dropping empty table with old schema: {TARGET_TABLE}")
            spark.sql(f"DROP TABLE {TARGET_TABLE}")
            table_exists = False
        else:
            raise Exception(f"Table {TARGET_TABLE} has {row_count} rows with incompatible schema. Manual migration required.")

if not table_exists:
    print(f"Creating target table: {TARGET_TABLE}")
    spark.createDataFrame([], target_schema).write.format("delta").saveAsTable(TARGET_TABLE)
    print("✓ Target table created")
else:
    print(f"Target table exists: {TARGET_TABLE}")

# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("aliases_extracted", IntegerType(), True),
    StructField("aliases_inserted", IntegerType(), True),
    StructField("aliases_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

# Create metadata table if not exists
if not spark.catalog.tableExists(METADATA_TABLE):
    print(f"Creating metadata table: {METADATA_TABLE}")
    spark.createDataFrame([], metadata_schema).write.format("delta").saveAsTable(METADATA_TABLE)
    print("✓ Metadata table created")
else:
    print(f"Metadata table exists: {METADATA_TABLE}")

In [0]:
print("Extracting company aliases from semantic layer...", end=" ")

# Load company mapping data and parse explanation_json to get original names
company_map_df = spark.table(SOURCE_MAP_TABLE)

# Parse explanation_json to extract the input company name (the alias)
alias_extract_df = company_map_df.select(
    F.get_json_object(F.col("explanation_json"), "$.input").alias("alias_name"),
    F.col("canonical_company_name"),
    F.col("normalization_method").alias("company_match_method"),
    F.col("normalization_confidence").alias("match_confidence")
).filter(
    F.col("alias_name").isNotNull() & 
    F.col("canonical_company_name").isNotNull()
).distinct()  # Remove duplicates

aliases_count = alias_extract_df.count()
print(f"✓ Extracted {aliases_count} unique company aliases")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(company_alias_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
print(f"Processing aliases into {TARGET_TABLE}... ", end="")

# Get existing aliases from target table
existing_aliases = spark.table(TARGET_TABLE).select(
    "alias_name",
    "canonical_company_name",
    "company_alias_sk"
)

# Join to company dimension to get company_sk
aliases_with_company = alias_extract_df.alias("a").join(
    spark.table(COMPANY_DIM_TABLE).select("company_name_canonical", "company_sk").alias("c"),
    F.col("a.canonical_company_name") == F.col("c.company_name_canonical"),
    "inner"
).select(
    F.col("a.alias_name"),
    F.col("a.canonical_company_name"),
    F.col("c.company_sk"),
    F.col("a.company_match_method"),
    F.col("a.match_confidence"),
    F.lit(True).alias("is_active")
)

# Left join with existing aliases to preserve surrogate keys
aliases_with_keys = aliases_with_company.alias("a").join(
    existing_aliases.alias("e"),
    (F.col("a.alias_name") == F.col("e.alias_name")) &
    (F.col("a.canonical_company_name") == F.col("e.canonical_company_name")),
    "left"
).select(
    F.col("a.*"),
    F.col("e.company_alias_sk").alias("existing_sk")
)

# Generate new surrogate keys for aliases without existing keys
window_spec = Window.orderBy("alias_name", "canonical_company_name")

aliases_final = aliases_with_keys.withColumn(
    "company_alias_sk",
    F.when(
        F.col("existing_sk").isNotNull(),
        F.col("existing_sk")
    ).otherwise(
        F.row_number().over(window_spec) + max_sk
    )
).withColumn(
    "match_confidence",
    F.col("match_confidence").cast(DoubleType())
).withColumn(
    "updated_at",
    run_timestamp
).select(
    "company_alias_sk",
    "alias_name",
    "canonical_company_name",
    "company_sk",
    "company_match_method",
    "match_confidence",
    "is_active",
    "updated_at"
)

# Create temp view for merge
aliases_final.createOrReplaceTempView("aliases_to_merge")

# Execute MERGE
try:
    merge_sql = f"""
    MERGE INTO {TARGET_TABLE} target
    USING aliases_to_merge source
    ON target.alias_name = source.alias_name
      AND target.canonical_company_name = source.canonical_company_name
    WHEN MATCHED THEN UPDATE SET
      target.company_sk = source.company_sk,
      target.company_match_method = source.company_match_method,
      target.match_confidence = source.match_confidence,
      target.is_active = source.is_active,
      target.updated_at = source.updated_at
    WHEN NOT MATCHED THEN INSERT (
      company_alias_sk,
      alias_name,
      canonical_company_name,
      company_sk,
      company_match_method,
      match_confidence,
      is_active,
      updated_at
    ) VALUES (
      source.company_alias_sk,
      source.alias_name,
      source.canonical_company_name,
      source.company_sk,
      source.company_match_method,
      source.match_confidence,
      source.is_active,
      source.updated_at
    )
    """
    
    spark.sql(merge_sql)
    
    # Get merge metrics
    new_aliases = aliases_final.join(existing_aliases, "company_alias_sk", "left_anti").count()
    updated_aliases = aliases_final.join(existing_aliases, "company_alias_sk", "inner").count()
    
    print(f"✓ Merge complete: {new_aliases} new, {updated_aliases} updated")
    
    # Write metadata
    metadata_record = spark.createDataFrame([(
        run_id,
        aliases_count,
        new_aliases,
        updated_aliases,
        force_full_refresh,
        datetime.now(),
        "success",
        None
    )], schema=metadata_schema)
    
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    # Return summary
    summary = {
        "status": "success",
        "run_id": run_id,
        "aliases_extracted": aliases_count,
        "aliases_inserted": new_aliases,
        "aliases_updated": updated_aliases,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(summary, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"\n✗ Error: {error_msg}")
    
    # Write error metadata
    metadata_record = spark.createDataFrame([(
        run_id,
        aliases_count,
        0,
        0,
        force_full_refresh,
        datetime.now(),
        "failed",
        error_msg
    )], schema=metadata_schema)
    
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate company alias dimension
SELECT 
  COUNT(*) as total_aliases,
  COUNT(DISTINCT company_sk) as distinct_companies,
  COUNT(DISTINCT alias_name) as distinct_alias_names,
  COUNT(DISTINCT canonical_company_name) as distinct_canonical_names,
  AVG(match_confidence) as avg_confidence,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_aliases
FROM workspace.warehouse.dim_company_alias;

In [0]:
%sql
-- Sample aliases grouped by company
SELECT 
  company_alias_sk,
  alias_name,
  canonical_company_name,
  company_sk,
  company_match_method,
  ROUND(match_confidence, 4) as match_confidence,
  is_active,
  updated_at
FROM workspace.warehouse.dim_company_alias
ORDER BY company_sk, alias_name
LIMIT 50;